# Validate All — strategy-level verdict matrix

Consolidates every `*_verdict.json` produced by `validate_strategy.ipynb`
into a single comparison matrix.  Use this notebook to answer
strategy-level questions without re-running validate per-instrument:

- *"Does the cross-sweep robust pick beat per-instrument bests on every
  instrument I tested?"*
- *"Which check is the most consistent red flag across instruments?"*
- *"Which combos pass on more instruments than others?"*

**Prerequisites:** Run `validate_strategy.ipynb` at least once with
`OVERRIDE_PARAMS=None` (auto-pick) and once with each combo you want
to compare.  Each run drops a JSON next to its snapshot HTML at
`reports/validate/{result_name}_verdict.json`.

**No new compute:** this notebook only reads the JSONs.  All the
heavy lifting (walk-forward, bootstrap, etc.) was done by the
individual validate runs.


## 1. Setup


### 1.1 Imports & filters

The matrix view loads every verdict JSON and filters to the runs you
care about.  Default: load everything, sort by instrument + pick +
timestamp.  Filter knobs are all optional.


In [ ]:
import pandas as pd
from IPython.display import display

# Add notebooks/ to sys.path so utils imports resolve.
import sys
sys.path.insert(0, str(__import__("pathlib").Path(".").resolve()))

from utils import (
    build_verdict_matrix,
    load_verdict_jsons,
    save_notebook_snapshot,
)

# ─────────────────────────────────────────────────────────────────
# Filters — applied to the loaded verdicts (None = no filter)
# ─────────────────────────────────────────────────────────────────
FILTER_INSTRUMENT : str | None = None     # e.g. "BTCUSDT-PERP.BINANCE"
FILTER_INTERVAL   : str | None = None     # e.g. "1d"
# When set, restrict to the most recent N verdicts per (instrument, pick)
# combo.  None = include all.  1 = "only the latest run for each".
LATEST_PER_PICK   : int | None = None

# ─────────────────────────────────────────────────────────────────
# Save behaviour
# ─────────────────────────────────────────────────────────────────
RESULT_NAME = "validate_all"
SAVE_ON_RUN_ALL = True

print("Imports OK")


### 1.2 Load all verdict JSONs

Reads every `reports/validate/*_verdict.json` written by
`validate_strategy.ipynb` and applies the optional filters from 1.1.


In [ ]:
verdicts = load_verdict_jsons()
print(f"Loaded {len(verdicts)} verdict JSON(s) from reports/validate/")

# Apply filters
if FILTER_INSTRUMENT is not None:
    verdicts = [v for v in verdicts if v.get("instrument_id") == FILTER_INSTRUMENT]
    print(f"  filtered to instrument={FILTER_INSTRUMENT!r}: {len(verdicts)} remain")
if FILTER_INTERVAL is not None:
    verdicts = [v for v in verdicts if v.get("bar_interval") == FILTER_INTERVAL]
    print(f"  filtered to interval={FILTER_INTERVAL!r}: {len(verdicts)} remain")

if LATEST_PER_PICK is not None and verdicts:
    # Keep only the N most-recent runs per (instrument, params) tuple.
    # Verdicts are already sorted newest-first by load_verdict_jsons.
    seen: dict = {}
    kept: list = []
    for v in verdicts:
        key = (
            v.get("instrument_id", ""),
            tuple(sorted((v.get("params") or {}).items())),
        )
        if seen.get(key, 0) < LATEST_PER_PICK:
            kept.append(v)
            seen[key] = seen.get(key, 0) + 1
    print(f"  kept latest {LATEST_PER_PICK} per (instrument, pick): {len(kept)} remain")
    verdicts = kept

if not verdicts:
    print()
    print("⚠️ No verdicts to display.  Run validate_strategy.ipynb")
    print("    with OVERRIDE_PARAMS = None or {fast: ..., slow: ...} to")
    print("    populate reports/validate/.")


## 2. Comparison matrix

One row per validate run.  Columns are the check icons (✅/⚠️/🚩) so
you can read horizontally to see which checks failed for which run.

The `pick` column is `"auto"` for runs where `OVERRIDE_PARAMS=None`
(per-instrument best) and the param string (e.g. `"fast=10, slow=20"`)
when an override was set.


In [ ]:
matrix = build_verdict_matrix(verdicts)
if matrix.empty:
    print("Empty matrix — no verdicts to compare.")
else:
    display(matrix)

    # Quick aggregate: how many runs landed in each verdict bucket?
    print()
    print("Verdict summary:")
    counts = matrix["verdict"].value_counts()
    for verdict_icon, count in counts.items():
        print(f"  {verdict_icon} {count}")


## 3. Per-check failure rate across runs

For each check (Plateau, Walk-forward, Param stability, …), what
fraction of the loaded runs were 🚩 / ⚠️ / ✅?  This identifies which
checks are the *consistent* red flags vs. which are noise.

A check that's 🚩 on most or all runs is telling you something
strategy-level (the param-stability / yearly-concentration patterns
in the EMA-cross test case).  A check that's mostly ✅ with one 🚩 is
instrument-specific noise.


In [ ]:
if not matrix.empty:
    # Identify which columns are check columns (between 'pick' and 'verdict')
    cols = list(matrix.columns)
    if "pick" in cols and "verdict" in cols:
        check_cols = cols[cols.index("pick") + 1 : cols.index("verdict")]
    else:
        check_cols = []

    if check_cols:
        rate_rows = []
        for c in check_cols:
            counts = matrix[c].value_counts()
            n_total = len(matrix)
            n_pass = int(counts.get("✅", 0))
            n_warn = int(counts.get("⚠️", 0))
            n_fail = int(counts.get("🚩", 0))
            rate_rows.append({
                "check": c,
                "✅": n_pass,
                "⚠️": n_warn,
                "🚩": n_fail,
                "% red flag": f"{n_fail / n_total * 100:.0f}%" if n_total else "—",
            })
        rate_df = pd.DataFrame(rate_rows).sort_values(
            "🚩", ascending=False,
        )
        print(f"Check outcomes across {len(matrix)} runs:")
        display(rate_df)


## 4. Single-run drill-down

Pick a row index from the matrix above and see its full check details
(the text that the validate notebook printed to stdout).  Useful when
the matrix flags something and you want the *why*.


In [ ]:
# ── Change this to inspect a different run ──
DRILL_INDEX = 0

if not verdicts or DRILL_INDEX >= len(verdicts):
    print("No verdict at that index.")
else:
    v = verdicts[DRILL_INDEX]
    params = v.get("params", {}) or {}
    param_str = ", ".join(f"{k}={vv}" for k, vv in params.items())
    print("=" * 60)
    print(f"  {v.get('instrument_id', '?')}  {v.get('bar_interval', '?')}")
    print(f"  {param_str}")
    print(f"  Source: {v.get('_source', '?')}")
    print(f"  Timestamp: {v.get('timestamp', '?')}")
    print("=" * 60)
    for c in v.get("checks", []):
        print(f"  {c['icon']} {c['name']:18s} {c['detail']}")
    verdict = v.get("verdict", {})
    print()
    print(f"  VERDICT: {verdict.get('icon', '')} {verdict.get('summary', '')}")
    print("=" * 60)


## 5. Save & cleanup


### 5.1 Save snapshot

Writes the executed notebook + rendered HTML to
`reports/notebooks/validate_all/` so the matrix is shareable.


In [ ]:
# Trailing semicolon suppresses Jupyter's auto-display of the
# returned Path (the helper already prints "Saved -> ..." for both
# .ipynb and .html).
save_notebook_snapshot(
    "validate_all.ipynb", RESULT_NAME,
    save_on_run_all=SAVE_ON_RUN_ALL,
    category="validate_all",
);


### 5.2 Cleanup

Nothing to dispose — this notebook only reads JSONs.


In [ ]:
print("Done.")


## 6. Scratchpad

Ad-hoc analysis goes here.


In [ ]:
# Scratchpad — your ad-hoc analysis goes here.
